<a href="https://colab.research.google.com/github/loan1/btxrd-stage1-semisup/blob/main/notebooks/3_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
%cd /content
!rm -rf btxrd-stage1-semisup
!git clone https://github.com/loan1/btxrd-stage1-semisup.git
%cd /content/btxrd-stage1-semisup
!git pull
!ls
!ls -la src

/content
Cloning into 'btxrd-stage1-semisup'...
remote: Enumerating objects: 55, done.
remote: Counting objects: 100% (55/55), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 55 (delta 22), reused 21 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (55/55), 33.47 KiB | 3.04 MiB/s, done.
Resolving deltas: 100% (22/22), done.
/content/btxrd-stage1-semisup
Already up to date.
notebooks  README.md  SAM_MEDSAM_BTXRD.ipynb  src
total 48
drwxr-xr-x 2 root root 4096 Mar  3 13:21 .
drwxr-xr-x 5 root root 4096 Mar  3 13:21 ..
-rw-r--r-- 1 root root 2712 Mar  3 13:21 btxrd_dataset.py
-rw-r--r-- 1 root root  460 Mar  3 13:21 config.py
-rw-r--r-- 1 root root 2727 Mar  3 13:21 data_utils.py
-rw-r--r-- 1 root root    1 Mar  3 13:21 .gitkeep
-rw-r--r-- 1 root root 2616 Mar  3 13:21 metrics.py
-rw-r--r-- 1 root root 5586 Mar  3 13:21 stage1_pipeline.py
-rw-r--r-- 1 root root 4718 Mar  3 13:21 train_utils.py
-rw-r--r-- 1 root root 1656 Mar  3 13:21 unet.py


In [3]:
import sys, torch
sys.path.insert(0, "/content/btxrd-stage1-semisup/src")

from config import RAW_DIR, PROC_DIR, PSEUDO_DIR, RUNS_DIR
from stage1_pipeline import train_and_report

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cuda


In [4]:
import os, glob, random

SPLIT_DIR = f"{PROC_DIR}/splits"
os.makedirs(SPLIT_DIR, exist_ok=True)

val_path  = os.path.join(SPLIT_DIR, "val.txt")
test_path = os.path.join(SPLIT_DIR, "test.txt")
train_path = os.path.join(SPLIT_DIR, "train.txt")

raw_images_dir = f"{RAW_DIR}/images"
all_ids = [os.path.basename(p) for p in glob.glob(os.path.join(raw_images_dir, "*"))]

print("Found images:", len(all_ids))
assert len(all_ids) > 0, "Không thấy ảnh trong RAW_DIR/images. Kiểm tra lại RAW_DIR trong src/config.py"

need_make = (not os.path.exists(val_path)) or (not os.path.exists(test_path))
if need_make:
    print("Splits missing -> creating train/val/test (70/10/20) ...")
    all_ids = sorted(all_ids)
    random.seed(2026)
    random.shuffle(all_ids)

    n = len(all_ids)
    n_train = int(0.70*n)
    n_val   = int(0.10*n)

    train_ids = all_ids[:n_train]
    val_ids   = all_ids[n_train:n_train+n_val]
    test_ids  = all_ids[n_train+n_val:]

    def write_list(path, ids):
        with open(path, "w") as f:
            for x in ids:
                f.write(x + "\n")

    write_list(train_path, train_ids)
    write_list(val_path, val_ids)
    write_list(test_path, test_ids)

    print("Saved:", train_path, val_path, test_path)
    print("train/val/test:", len(train_ids), len(val_ids), len(test_ids))
else:
    print("Splits exist:", val_path, test_path)

print("Split dir files:", os.listdir(SPLIT_DIR))

Found images: 3746
Splits exist: /content/drive/MyDrive/BTXRD/processed/splits/val.txt /content/drive/MyDrive/BTXRD/processed/splits/test.txt
Split dir files: ['train.txt', 'train_labeled.txt', 'test.txt', 'val.txt', 'train_unlabeled.txt', 'train_tumor_all.txt', 'train_normal_all.txt']


In [5]:
import os

gt_dir = f"{PROC_DIR}/masks_gt"
assert os.path.exists(gt_dir), f"Thiếu {gt_dir}"

pseudo_dir = f"{PSEUDO_DIR}/sam_box_oracle"
print("GT masks dir:", gt_dir, "exists:", os.path.exists(gt_dir))
print("Pseudo dir:", pseudo_dir, "exists:", os.path.exists(pseudo_dir))

# không bắt buộc pseudo để chạy supervised

GT masks dir: /content/drive/MyDrive/BTXRD/processed/masks_gt exists: True
Pseudo dir: /content/drive/MyDrive/BTXRD/pseudo/sam_box_oracle exists: True


In [6]:
result_sup, info_sup, chosen_sup = train_and_report(
    raw_dir=RAW_DIR,
    proc_dir=PROC_DIR,
    pseudo_dir=PSEUDO_DIR,
    runs_dir=RUNS_DIR,
    exp_name="p10_supervised_pw10_auto",
    run_mode="supervised",
    p=0.10,
    device=device,
    pos_weight=10.0,
    epochs=20,
    sampling="auto_group_balance",
    thr_select=True,
    max_fp_rate=0.6,
    resume=True
)

print("INFO:", info_sup)
print("Chosen thr:", chosen_sup)
print("RESULT:", result_sup)

Epoch 01 | loss=0.7061 | val_soft_dice_tumor=0.0665 | val_dice_tumor@0.5=0.0000


Epoch 02 | loss=0.6434 | val_soft_dice_tumor=0.0543 | val_dice_tumor@0.5=0.0000


Epoch 03 | loss=0.6231 | val_soft_dice_tumor=0.0689 | val_dice_tumor@0.5=0.0000


Epoch 04 | loss=0.6135 | val_soft_dice_tumor=0.0508 | val_dice_tumor@0.5=0.0000


Epoch 05 | loss=0.6187 | val_soft_dice_tumor=0.0725 | val_dice_tumor@0.5=0.0000


...

Epoch 16 | loss=0.5638 | val_soft_dice_tumor=0.0997 | val_dice_tumor@0.5=0.0623


Epoch 17 | loss=0.5598 | val_soft_dice_tumor=0.0863 | val_dice_tumor@0.5=0.0342


Epoch 18 | loss=0.5489 | val_soft_dice_tumor=0.0226 | val_dice_tumor@0.5=0.0048


Epoch 19 | loss=0.5522 | val_soft_dice_tumor=0.1176 | val_dice_tumor@0.5=0.1091


Epoch 20 | loss=0.5378 | val_soft_dice_tumor=0.1739 | val_dice_tumor@0.5=0.2149
Done. Best val_soft_dice_tumor: 0.17385014441485205 best: /content/drive/MyDrive/BTXRD/runs/p10_supervised_pw10_auto/best.pth last: /content/drive/MyDrive/BTXRD/runs/p10_supervised_pw10_auto/last.pt
INFO: {'val': 375, 'test': 750, 'train_tumor': 1307, 'train_normal': 1314, 'labeled_tumor': 131, 'unlabeled_tumor': 1176, 'sampling': 'auto_group_balance', 'normal_weight_per_sample': 0.09969558599695585, 'cache': {'cached': True, 'cache_tumor': '/content/drive/MyDrive/BTXRD/processed/splits/train_tumor_all.txt', 'cache_normal': '/content/drive/MyDrive/BTXRD/processed/splits/train_normal_all.txt'}}
Chosen thr: {'thr': 0.9, 'dice_all': 0.30880465389539796, 'iou_all': 0.29538361689386267, 'dice_tumor': 0.08134898591338201, 'iou_tumor': 0.05378515862644193, 'normal_images': 188, 'fp_normals': 87, 'fp_rate': 0.4627659574468085, 'avg_pred_area_pixels': 91.62336837484482}
RESULT: {'exp': 'p10_supervised_pw10_auto', 'm

In [ ]:
result_semi, info_semi, chosen_semi = train_and_report(
    raw_dir=RAW_DIR,
    proc_dir=PROC_DIR,
    pseudo_dir=PSEUDO_DIR,
    runs_dir=RUNS_DIR,
    exp_name="p10_semi_sam_pw10_auto",
    run_mode="semi",
    p=0.10,
    device=device,
    pos_weight=10.0,
    epochs=20,
    sampling="auto_group_balance",
    pseudo_subdir="sam_box_oracle",
    thr_select=True,
    max_fp_rate=0.6,
    resume=True
)

print("INFO:", info_semi)
print("Chosen thr:", chosen_semi)
print("RESULT:", result_semi)

Epoch 01 | loss=0.6832 | val_soft_dice_tumor=0.0684 | val_dice_tumor@0.5=0.0000


Train 2:  49%|████▉     | 80/164 [02:07<01:29,  1.07s/it]